# Simulation Code T = 2

## Import libraries

In [1]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.estimateDiDLinear import estimateDiDLinear
import torch
import pandas as pd
import time
from torch.distributions import Normal
from utils.dgp import DiD_DGP

## Estimation Settings

In [2]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : 1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : 1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

##### Simulation settings:

In [3]:
torch.manual_seed(123);

# Parameters
N = 300
tmax = 2

# Higher dimensions = more sparsity
dimX = 3
dimZ = 3

##### Define treatment probability function

In [4]:
# Bounds (only for truncated distributions)
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)


In [5]:
treatment_probability_func = truncated_logistic

In [6]:
dgp = DiD_DGP(g = logistic)
data = dgp.generate(1000, 0)

##### Coefficients

In [7]:
beta_0_Y1 = 1
beta_Z_Y1 = torch.tensor([1, 1] + [0] * (dimZ - 2), dtype=torch.float32).reshape(-1,1) 
beta_X1_Y1 = torch.tensor([1, 1] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 

beta_0_pi = 0
beta_Z_pi = torch.tensor([1, -1] + [0] * (dimZ - 2), dtype=torch.float32).reshape(-1,1) 
beta_X1_pi = torch.tensor([-1, 1] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 
beta_Y1_pi = torch.tensor([0.5], dtype=torch.float32).reshape(-1,1) 

beta_0_Y2 = 1
beta_0_Y2_D = 1

beta_Z_Y2 = torch.tensor([0.5, 1.5] + [0] * (dimZ - 2), dtype=torch.float32).reshape(-1,1) 
beta_X1_Y2 = torch.tensor([0.5, 0.5] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 
beta_X2_Y2 = torch.tensor([1.5, 1.5] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 

beta_Z_Y2_D = torch.tensor([1, 1] + [0] * (dimZ - 2), dtype=torch.float32).reshape(-1,1) 
beta_X1_Y2_D = torch.tensor([0.5, 0.5] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 
beta_X2_Y2_D = torch.tensor([1, 1] + [0] * (dimX - 2), dtype=torch.float32).reshape(-1,1) 

##### Simulate data

In [8]:
# Period 1
Z_all = torch.randn(N * tmax, dimZ) 
X1_all = torch.randn(N * tmax, dimX) 
Y1_all = beta_0_Y1 + Z_all @ beta_Z_Y1 + X1_all @ beta_X1_Y1 + torch.randn(N * tmax,1)

# Treatment
pi_all = treatment_probability_func(beta_0_pi + Z_all @ beta_Z_pi + X1_all @ beta_X1_pi + Y1_all @ beta_Y1_pi).reshape(-1,1) # Generated treatment probablities
pi_all = 0.5 * torch.ones(N * tmax, 1)
D_all = torch.bernoulli(pi_all).int().reshape(-1,1) # Generate treatment assignment period 1

# Period 2 covariates
delta1_all = torch.randn(N * tmax,1) # Change from X1 to X2 if treated in period 1
delta2_all = torch.randn(N * tmax, dimX) # Change from X1 to X2 always

X2_all = X1_all + D_all * (1 + delta1_all) + delta2_all # Period 2 covariates
X2_1_all = X1_all + 1 + delta1_all + delta2_all # Period 2 covariates if treated in period 1
X2_0_all = X1_all + delta2_all # Period 2 covariates if untreated in period 1

# Period 2 outcome
g_all_0 = beta_0_Y2 + Z_all @ beta_Z_Y2 + X1_all @ beta_X1_Y2 + X2_0_all @ beta_X2_Y2
g_all_1 = g_all_0 + (beta_0_Y2_D + beta_0_Y2_D + Z_all @ beta_Z_Y2_D + X1_all @ beta_X1_Y2_D + X2_1_all @ beta_X2_Y2_D)
zeta_all = torch.randn(N * tmax,1) # Noise to outcome

Y2_all = (1 - D_all) * g_all_0 + D_all * g_all_1 + zeta_all

In [9]:
torch.mean(Y2_all[(D_all == 1).squeeze()] - Y1_all[(D_all == 1).squeeze()])

tensor(3.8063)

#### True values:

In [10]:

# True ATT:

true_ATT = torch.mean(g_all_1[(D_all == 0).squeeze()] - g_all_0[(D_all == 0).squeeze()])
true_ATT


tensor(4.2601)

## Estimation:

#### Estimation settings

In [11]:
folds = 4

time0 = time.time()

#### Estimation 

In [12]:
pred_theta = torch.zeros(tmax,18)
pred_sig = torch.zeros(tmax,18)

RR1 = torch.zeros(N,tmax,18)
RR2 = torch.zeros(N,tmax,18)
f1 = torch.zeros(N,tmax,18)
f2 = torch.zeros(N,tmax,18)

for t in range(0,1):

    # Get data for current iteration
    X1, X2 = X1_all[(t) * N : (t+1) * N, :], X2_all[(t) * N : (t+1) * N, :]
    D, Z = D_all[(t) * N : (t+1) * N], Z_all[(t) * N : (t+1) * N]
    Y1, Y2 = Y1_all[(t) * N : (t+1) * N, :], Y2_all[(t) * N : (t+1) * N, :]


    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    # pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    # mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    # if d[0,0] == 1:
    #     RR2[:,t,:1], RR1[:,t,:1] = D1 * D2 / (pi1 * pi2_1), D1 / pi1 
    #     theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_1   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_1)
    #     pred_theta[t,0] = torch.mean(theta_hat)
    #     pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    # elif d[0,0] == 0:
    #     RR2[:,t,:1], RR1[:,t,:1] = (1 - D1) * (1 - D2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - D1) / (1 - pi1)
    #     theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_0   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_0)
    #     pred_theta[t,0] = torch.mean(theta_hat)
    #     pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    #---------------------------------------------------------------------------
    # Estimator 2: Linear (not doubly robust) estimation from Cataneo et al.


    #---------------------------------------------------------------------------
    # # Estimator 3: LASSO, RF, Net

    # pred_theta[t,2:], pred_sig[t,2:], RR1[:,t,2:], RR2[:,t,2:] = estimateDynamicRiesz_all(Y1, Y2, D, Z, X1, X2, folds)

    #pred_theta[t,1], pred_sig[t,1] = estimateDiDLinear(Y1, Y2, D, Z, X1, X2)

    out1= estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                     method_a = "LASSO", rf_a_settings = rf_a_settings,
                                                                        method_f = "LASSO", rf_f_settings = rf_f_settings, seed = 0)
    out2= estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                     method_a = "LASSO", rf_a_settings = rf_a_settings,
                                                                        method_f = "LASSO", rf_f_settings = rf_f_settings, seed = 0)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  25.832457065582275
Finished. Total time:  25.832573890686035


In [13]:
print(out1[0])
print(out2[0])

tensor(4.1915)
tensor(4.1915)


## Compute statistics

#### Get true value

In [ ]:
true_value = true_ATT

#### Compute statistics

In [14]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO_LASSO", "LASSO_RF", "LASSO_Net", "LASSO_RF1", "RF_LASSO", "RF_RF", "RF_Net", "RF_RF1", "Net_LASSO", "Net_RF", "Net_Net", "Net_RF1", "RF0_LASSO", "RF0_RF", "RF0_Net", "RF0_RF1"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

NameError: name 'true_value' is not defined

#### Print results

In [ ]:
df = pd.DataFrame(data)
print(df)

         Method      Bias      RMSE  Coverage  Interval Length
0        Oracle -3.972229  3.972229       0.0         0.000000
1        Bradic -0.041351  0.043922       1.0         0.493939
2   LASSO_LASSO -3.972229  3.972229       0.0         0.000000
3      LASSO_RF -0.059679  0.060579       1.0         0.899480
4     LASSO_Net -3.972229  3.972229       0.0         0.000000
5     LASSO_RF1 -3.972229  3.972229       0.0         0.000000
6      RF_LASSO -3.972229  3.972229       0.0         0.000000
7         RF_RF -3.972229  3.972229       0.0         0.000000
8        RF_Net -3.972229  3.972229       0.0         0.000000
9        RF_RF1 -3.972229  3.972229       0.0         0.000000
10    Net_LASSO -3.972229  3.972229       0.0         0.000000
11       Net_RF -3.972229  3.972229       0.0         0.000000
12      Net_Net -3.972229  3.972229       0.0         0.000000
13      Net_RF1 -3.972229  3.972229       0.0         0.000000
14    RF0_LASSO -3.972229  3.972229       0.0         0

In [ ]:
pred_theta

tensor([[0.0000, 3.9161, 0.0000, 3.9021, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 3.9457, 0.0000, 3.9230, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [ ]:
# check_t = 0
# Compare RR:
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
